# 15 — Held-out success and failure case diagnostics

This notebook responds directly to the supervisor request to show **successful and failure cases** for both PTAL and EPC.

It does **not retrain any model**. It uses the frozen borough-held-out out-of-fold (OOF) predictions already produced by Notebook 07 and the frozen analytical sample. Case selection is deterministic and based on prediction errors, so the displayed examples are not hand-picked for visual appeal.

Primary matched comparisons:
- **PTAL:** spatial controls → spatial controls + all representations + Street View metadata.
- **EPC:** compact controls → compact controls + all representations + Street View metadata.

Selection rule within each task:
1. **Success candidates:** top decile of baseline-to-augmented absolute-error reduction and bottom quartile of augmented absolute error.
2. Select two success cases from distinct boroughs where possible.
3. **Large-error failure:** largest augmented absolute error not already selected.
4. **Worsening failure:** largest deterioration relative to the baseline (`error_reduction < 0`) not already selected.
5. Prefer distinct boroughs across the four displayed cases where the deterministic candidate ordering permits it.

The diagnostic is descriptive and hypothesis-generating. It does not identify the visual feature(s) used by a black-box encoder and must not be interpreted as causal feature attribution.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, math, ast, re, zipfile
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw

BASE = Path('/content/drive/MyDrive/GEOG0105')
FINAL_OUT = BASE / 'Outputs' / 'final_pipeline'
MODEL_DIR = FINAL_OUT / 'models'
TABLE_DIR = FINAL_OUT / 'tables'
AUDIT_DIR = FINAL_OUT / 'audit'
FIG_DIR = FINAL_OUT / 'figures'
RAW_DIR = BASE / 'Raw Data'

PRED_PATH = MODEL_DIR / '07_incremental_predictions.parquet'
COMMON_PATH = TABLE_DIR / 'common_sample_final.parquet'
AERIAL_INDEX_PATH = BASE / 'Outputs' / 'tables' / 'aerial_crop_source_index_all.csv'
STREET_INVENTORY_PATH = BASE / 'Outputs' / 'tables' / 'streetview_image_inventory.csv'
STREET_ZIP_PATH = RAW_DIR / 'Street_view' / 'Streetviews.zip'

OUT_SELECTION = AUDIT_DIR / '15_success_failure_case_selection.csv'
OUT_AUDIT = AUDIT_DIR / '15_success_failure_case_audit.json'
OUT_PNG = FIG_DIR / 'Figure_16_held_out_success_failure_cases.png'
OUT_PDF = FIG_DIR / 'Figure_16_held_out_success_failure_cases.pdf'

for p in [PRED_PATH, COMMON_PATH]:
    assert p.exists(), f'Missing frozen input: {p}'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Frozen predictions:', PRED_PATH)
print('Common sample:', COMMON_PATH)


In [ ]:
pred = pd.read_parquet(PRED_PATH)
common = pd.read_parquet(COMMON_PATH)

required_pred = {'sample_id','task','model_id','outer_fold','borough_code','y_true','y_pred'}
missing = required_pred - set(pred.columns)
assert not missing, f'Prediction columns missing: {sorted(missing)}'
assert set(common['task'].unique()) == {'PTAL','EPC'}
assert len(common) == 26597
assert common['sample_id'].is_unique

print('Prediction rows:', len(pred))
print('Prediction model IDs:', pred['model_id'].nunique())
display(pd.DataFrame({'column': common.columns}))


In [ ]:
PRIMARY = {
    'PTAL': {
        'baseline': 'PTAL_spatial_baseline',
        'augmented': 'PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata',
    },
    'EPC': {
        'baseline': 'EPC_controls_sparse',
        'augmented': 'EPC_controls_sparse__plus__All_representations_plus_SV_metadata',
    },
}

def matched_task_predictions(task):
    spec = PRIMARY[task]
    sub = pred[(pred.task == task) & (pred.model_id.isin([spec['baseline'], spec['augmented']]))].copy()
    piv = sub.pivot_table(
        index=['sample_id','task','outer_fold','borough_code','y_true'],
        columns='model_id',
        values='y_pred',
        aggfunc='first'
    ).reset_index()
    assert spec['baseline'] in piv and spec['augmented'] in piv
    piv = piv.rename(columns={
        spec['baseline']: 'pred_baseline',
        spec['augmented']: 'pred_augmented'
    })
    piv['abs_err_baseline'] = (piv.y_true - piv.pred_baseline).abs()
    piv['abs_err_augmented'] = (piv.y_true - piv.pred_augmented).abs()
    piv['error_reduction'] = piv.abs_err_baseline - piv.abs_err_augmented
    piv['signed_error_augmented'] = piv.pred_augmented - piv.y_true
    return piv

matched = pd.concat([matched_task_predictions('PTAL'), matched_task_predictions('EPC')], ignore_index=True)
assert matched['sample_id'].nunique() == 26597
display(matched.groupby('task')[['abs_err_baseline','abs_err_augmented','error_reduction']].describe())


In [ ]:
def greedy_distinct_borough(frame, n, used=None):
    used = set() if used is None else set(used)
    rows = []
    for row in frame.itertuples(index=False):
        if row.borough_code in used:
            continue
        rows.append(row)
        used.add(row.borough_code)
        if len(rows) == n:
            break
    if len(rows) < n:
        # deterministic fallback if borough diversity cannot be satisfied
        selected_ids = {r.sample_id for r in rows}
        for row in frame.itertuples(index=False):
            if row.sample_id in selected_ids:
                continue
            rows.append(row)
            if len(rows) == n:
                break
    return pd.DataFrame(rows), used

def select_cases(task):
    g = matched[matched.task == task].copy()
    q_red90 = g.error_reduction.quantile(.90)
    q_err25 = g.abs_err_augmented.quantile(.25)

    success_pool = g[
        (g.error_reduction >= q_red90) &
        (g.abs_err_augmented <= q_err25)
    ].sort_values(
        ['error_reduction','abs_err_augmented','sample_id'],
        ascending=[False, True, True],
        kind='mergesort'
    )
    succ, used = greedy_distinct_borough(success_pool, 2)
    succ = succ.copy()
    succ['case_type'] = ['Success 1','Success 2'][:len(succ)]

    selected = set(succ.sample_id)

    large_pool = g[~g.sample_id.isin(selected)].sort_values(
        ['abs_err_augmented','sample_id'],
        ascending=[False, True],
        kind='mergesort'
    )
    large = large_pool.iloc[[0]].copy()
    large['case_type'] = 'Failure: large held-out error'
    selected.update(large.sample_id)
    used.update(large.borough_code)

    worsening_pool = g[
        (~g.sample_id.isin(selected)) &
        (g.error_reduction < 0)
    ].sort_values(
        ['error_reduction','abs_err_augmented','sample_id'],
        ascending=[True, False, True],
        kind='mergesort'
    )
    # Prefer a new borough for the worsening case.
    worsening_new = worsening_pool[~worsening_pool.borough_code.isin(used)]
    worsening = (worsening_new if len(worsening_new) else worsening_pool).iloc[[0]].copy()
    worsening['case_type'] = 'Failure: augmentation worsened baseline'

    out = pd.concat([succ, large, worsening], ignore_index=True)
    assert len(out) == 4, (task, len(out))

    thresholds = {
        'success_error_reduction_p90': float(q_red90),
        'success_augmented_abs_error_p25': float(q_err25),
        'failure_large_error_rule': 'maximum augmented absolute error among unselected cases',
        'failure_worsening_rule': 'most negative baseline-to-augmented error reduction among unselected cases; prefer new borough',
    }
    return out, thresholds

selected_parts = []
thresholds = {}
for task in ['PTAL','EPC']:
    s, th = select_cases(task)
    thresholds[task] = th
    selected_parts.append(s)

selected = pd.concat(selected_parts, ignore_index=True)
selected = selected.merge(common, on=['sample_id','task'], how='left', suffixes=('','_sample'), validate='one_to_one')
selected.to_csv(OUT_SELECTION, index=False)

display(selected[['task','case_type','sample_id','borough_code','y_true','pred_baseline','pred_augmented',
                  'abs_err_baseline','abs_err_augmented','error_reduction']])

audit = {
    'source_predictions': str(PRED_PATH),
    'source_common_sample': str(COMMON_PATH),
    'primary_models': PRIMARY,
    'selection_rule': {
        'success': 'top decile error reduction AND bottom quartile augmented absolute error; two cases, distinct boroughs if possible',
        'large_error_failure': 'maximum augmented absolute error among remaining cases',
        'worsening_failure': 'most negative error reduction among remaining cases, preferring a new borough',
    },
    'thresholds': thresholds,
    'n_cases': int(len(selected)),
}
OUT_AUDIT.write_text(json.dumps(audit, indent=2), encoding='utf-8')
print('Saved:', OUT_SELECTION)
print('Saved:', OUT_AUDIT)


## Visual case figure

The code below attempts to retrieve the task-specific aerial crop and the nearest readable archived Street View image for each selected location.

If a visual source cannot be resolved for a case, the notebook keeps the case and shows a labelled placeholder. A missing image therefore cannot silently change which case was selected.


In [ ]:
# Resolve BNG coordinates in the selected common-sample columns.
def find_xy_columns(frame):
    candidates = [
        ('x','y'), ('bng_x','bng_y'), ('easting','northing'),
        ('EASTING','NORTHING'), ('x_bng','y_bng')
    ]
    for xcol,ycol in candidates:
        if xcol in frame.columns and ycol in frame.columns:
            return xcol,ycol
    # common aliases occasionally used in the project
    xhits = [c for c in frame.columns if c.lower() in {'x_coord','coord_x','os_easting'}]
    yhits = [c for c in frame.columns if c.lower() in {'y_coord','coord_y','os_northing'}]
    if xhits and yhits:
        return xhits[0], yhits[0]
    raise KeyError('Could not identify BNG x/y columns in common_sample_final.parquet.')

XCOL, YCOL = find_xy_columns(selected)
selected[XCOL] = pd.to_numeric(selected[XCOL], errors='coerce')
selected[YCOL] = pd.to_numeric(selected[YCOL], errors='coerce')
assert selected[[XCOL,YCOL]].notna().all().all()
print('Using coordinate columns:', XCOL, YCOL)


In [ ]:
# Optional visual-source setup.
visual_sources_ok = AERIAL_INDEX_PATH.exists() and STREET_INVENTORY_PATH.exists() and STREET_ZIP_PATH.exists()
print({
    'aerial_index': AERIAL_INDEX_PATH.exists(),
    'street_inventory': STREET_INVENTORY_PATH.exists(),
    'street_zip': STREET_ZIP_PATH.exists(),
})

if visual_sources_ok:
    import rasterio
    from rasterio.enums import Resampling
    from rasterio.merge import merge
    from rasterio.windows import from_bounds
    from sklearn.neighbors import BallTree

    aerial_index = pd.read_csv(AERIAL_INDEX_PATH, low_memory=False)
    for c in ['x','y']:
        if c in aerial_index:
            aerial_index[c] = pd.to_numeric(aerial_index[c], errors='coerce')

    street_inventory = pd.read_csv(STREET_INVENTORY_PATH, low_memory=False)
    street_inventory['_x'] = pd.to_numeric(street_inventory['sv_x'], errors='coerce')
    street_inventory['_y'] = pd.to_numeric(street_inventory['sv_y'], errors='coerce')

    # Convert lon/lat inventory coordinates to BNG if necessary.
    if street_inventory['_x'].abs().median() < 10000:
        import geopandas as gpd
        sg = gpd.GeoSeries(
            gpd.points_from_xy(street_inventory._x, street_inventory._y),
            crs='EPSG:4326'
        ).to_crs('EPSG:27700')
        street_inventory['_x'] = sg.x
        street_inventory['_y'] = sg.y

    street_inventory = street_inventory.dropna(subset=['_x','_y']).copy()
    if 'has_image' in street_inventory:
        street_inventory = street_inventory[pd.to_numeric(street_inventory.has_image, errors='coerce').eq(1)].copy()

    with zipfile.ZipFile(STREET_ZIP_PATH) as z:
        zip_members = set(z.namelist())
    street_inventory = street_inventory[
        street_inventory.image_in_zip.astype(str).isin(zip_members)
    ].copy()

    sv_tree = BallTree(street_inventory[['_x','_y']].to_numpy(), leaf_size=40)

def parse_tile_paths(row):
    paths = []
    for col in ['tile_paths','tile_path']:
        if col not in row.index:
            continue
        value = row.get(col, '')
        if pd.isna(value) or not str(value).strip():
            continue
        value = str(value).strip()
        try:
            parsed = ast.literal_eval(value)
            paths.extend(parsed if isinstance(parsed,(list,tuple)) else [parsed])
        except Exception:
            paths.extend([v.strip() for v in re.split(r'[|;]', value) if v.strip()])
    return list(dict.fromkeys(Path(p) for p in paths if Path(p).exists()))

def aerial_crop_for_point(sample_id, x, y, support_m, output_px=500):
    if not visual_sources_ok:
        return None
    # Prefer exact sample_id match when present; otherwise nearest indexed centre.
    if 'sample_id' in aerial_index.columns:
        exact = aerial_index[aerial_index.sample_id.astype(str).eq(str(sample_id))]
    else:
        exact = aerial_index.iloc[0:0]

    if len(exact):
        candidates = exact
    elif {'x','y'}.issubset(aerial_index.columns):
        work = aerial_index.dropna(subset=['x','y']).copy()
        d = np.hypot(work.x - x, work.y - y)
        candidates = work.loc[d.nsmallest(20).index]
    else:
        return None

    half = support_m / 2
    bounds = (x-half, y-half, x+half, y+half)

    for _, row in candidates.iterrows():
        paths = parse_tile_paths(row)
        if not paths:
            continue
        datasets = []
        try:
            datasets = [rasterio.open(p) for p in paths]
            if len(datasets) == 1:
                src = datasets[0]
                win = from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
                bands = list(range(1, min(src.count,3)+1))
                arr = src.read(
                    bands, window=win,
                    out_shape=(len(bands), output_px, output_px),
                    resampling=Resampling.bilinear,
                    boundless=True, fill_value=255
                )
            else:
                arr,_ = merge(
                    datasets, bounds=bounds,
                    indexes=list(range(1,min(datasets[0].count,3)+1))
                )
            arr = np.moveaxis(arr[:3],0,-1)
            if arr.shape[-1] == 1:
                arr = np.repeat(arr,3,axis=-1)
            if arr.dtype != np.uint8:
                lo,hi = np.nanpercentile(arr,[1,99])
                arr = np.clip((arr-lo)/(hi-lo+1e-9)*255,0,255).astype(np.uint8)
            return np.asarray(ImageOps.fit(Image.fromarray(arr).convert('RGB'), (output_px,output_px)))
        except Exception:
            continue
        finally:
            for ds in datasets:
                try: ds.close()
                except Exception: pass
    return None

def nearest_street_image(x, y, radius_m, output_px=230):
    if not visual_sources_ok:
        return None, None
    ind, dist = sv_tree.query_radius([[x,y]], r=radius_m, return_distance=True, sort_results=True)
    if len(ind[0]) == 0:
        return None, None
    with zipfile.ZipFile(STREET_ZIP_PATH) as archive:
        for idx,d in zip(ind[0], dist[0]):
            member = str(street_inventory.iloc[int(idx)].image_in_zip)
            try:
                im = Image.open(BytesIO(archive.read(member))).convert('RGB')
                im = ImageOps.fit(ImageOps.exif_transpose(im), (output_px,output_px))
                return np.asarray(im), float(d)
            except Exception:
                continue
    return None, None


In [ ]:
# Build the final 2×4 case figure.
fig, axes = plt.subplots(2, 4, figsize=(15.5, 8.2))

for row_i, task in enumerate(['PTAL','EPC']):
    task_cases = selected[selected.task == task].copy()
    order = ['Success 1','Success 2','Failure: large held-out error','Failure: augmentation worsened baseline']
    task_cases['__order'] = task_cases.case_type.map({k:i for i,k in enumerate(order)})
    task_cases = task_cases.sort_values('__order')

    support = 300 if task == 'PTAL' else 150
    sv_radius = 300 if task == 'PTAL' else 150

    for col_i, case in enumerate(task_cases.itertuples(index=False)):
        ax = axes[row_i, col_i]
        x = float(getattr(case, XCOL))
        y = float(getattr(case, YCOL))

        aerial = aerial_crop_for_point(case.sample_id, x, y, support)
        if aerial is not None:
            ax.imshow(aerial)
        else:
            ax.set_facecolor('0.94')
            ax.text(.5,.54,'Aerial crop unavailable',ha='center',va='center',transform=ax.transAxes,fontsize=9)

        sv, dist = nearest_street_image(x, y, sv_radius)
        if sv is not None:
            inset = ax.inset_axes([.03,.03,.31,.31])
            inset.imshow(sv)
            inset.set_xticks([]); inset.set_yticks([])
            inset.set_title(f'SV {dist:.0f} m', fontsize=6, pad=1.5)

        err_delta = case.error_reduction
        relation = 'improved' if err_delta >= 0 else 'worsened'
        borough = str(case.borough_code)

        ax.set_title(
            f'{task} — {case.case_type}\n'
            f'Observed {case.y_true:.1f} | control {case.pred_baseline:.1f} | reps {case.pred_augmented:.1f}\n'
            f'|error| {case.abs_err_baseline:.1f} → {case.abs_err_augmented:.1f} ({relation} {abs(err_delta):.1f})',
            fontsize=8.0, loc='left'
        )
        ax.text(.98,.02,borough,ha='right',va='bottom',transform=ax.transAxes,fontsize=6.5,
                bbox=dict(boxstyle='round,pad=.2',facecolor='white',alpha=.8,edgecolor='none'))
        ax.set_xticks([]); ax.set_yticks([])

fig.suptitle(
    'Held-out success and failure cases from the primary borough-held-out comparison',
    fontsize=13, y=.995
)
fig.text(
    .5,.008,
    'Cases are selected deterministically from frozen OOF predictions; visual inspection is diagnostic, not feature attribution.',
    ha='center', fontsize=8
)
fig.tight_layout(rect=(0,.035,1,.97), w_pad=1.0, h_pad=1.2)

fig.savefig(OUT_PNG, dpi=240, bbox_inches='tight')
fig.savefig(OUT_PDF, bbox_inches='tight')
plt.show()

print('Saved:', OUT_PNG)
print('Saved:', OUT_PDF)


In [ ]:
# Final audit checks
sel = pd.read_csv(OUT_SELECTION)
assert len(sel) == 8
assert set(sel.task) == {'PTAL','EPC'}
assert sel.groupby('task').size().eq(4).all()
assert np.isfinite(sel[['y_true','pred_baseline','pred_augmented','abs_err_baseline',
                        'abs_err_augmented','error_reduction']].to_numpy()).all()

print('\nFINAL OUTPUTS')
for p in [OUT_SELECTION, OUT_AUDIT, OUT_PNG, OUT_PDF]:
    print(p, 'exists=', p.exists(), 'bytes=', p.stat().st_size if p.exists() else None)
